<a href="https://colab.research.google.com/github/Coobaltoo13/milling-tool-life-fusion/blob/main/07_tsfresh_feature_extraction_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile
import os

# Ruta del ZIP (ajusta según dónde lo tengas)
# Si lo subiste a Drive:
ruta_zip = '/content/drive/MyDrive/raw_data.zip'

# Si lo descargaste directo a Colab:
# ruta_zip = '/content/raw_data.zip'

with zipfile.ZipFile(ruta_zip, 'r') as z:
    archivos = z.namelist()
    print(f"Total de archivos: {len(archivos)}")
    print("\nPrimeros 20 archivos:")
    for a in archivos[:20]:
        print(a)

    # Ver tamaño total descomprimido
    total_size = sum(info.file_size for info in z.infolist())
    print(f"\nTamaño total descomprimido: {total_size / (1024**3):.2f} GB")

Total de archivos: 969

Primeros 20 archivos:
raw_data/
raw_data/P002_F01_C1.csv
raw_data/P002_F01_C2.csv
raw_data/P002_F01_C3.csv
raw_data/P002_F01_C4.csv
raw_data/P002_F02_C1.csv
raw_data/P002_F02_C2.csv
raw_data/P002_F02_C3.csv
raw_data/P002_F02_C4.csv
raw_data/P002_F03_C1.csv
raw_data/P002_F03_C2.csv
raw_data/P002_F03_C3.csv
raw_data/P002_F03_C4.csv
raw_data/P002_F04_C1.csv
raw_data/P002_F04_C2.csv
raw_data/P002_F04_C3.csv
raw_data/P002_F04_C4.csv
raw_data/P002_F05_C1.csv
raw_data/P002_F05_C2.csv
raw_data/P002_F05_C3.csv

Tamaño total descomprimido: 104.13 GB


Ver que contiene el archivo


In [3]:
import zipfile
import pandas as pd

# Ruta del ZIP (ajusta según dónde lo tengas)
ruta_zip = '/content/drive/MyDrive/raw_data.zip'  # o '/content/raw_data.zip'

with zipfile.ZipFile(ruta_zip, 'r') as z:
    # Tomar un archivo de ejemplo
    archivo_ejemplo = 'raw_data/P002_F01_C1.csv'

    with z.open(archivo_ejemplo) as f:
        df = pd.read_csv(f)
        print("Forma del archivo:", df.shape)
        print("\nPrimeras 5 filas:")
        print(df.head())
        print("\nNombres de columnas:")
        print(df.columns.tolist())
        print("\nTipos de datos:")
        print(df.dtypes)

Forma del archivo: (2698201, 22)

Primeras 5 filas:
   Timestamps - Acc  Accelerometer - Spindle +Y  Accelerometer - Spindle -Z  \
0              0.00                      -442.0                      -134.0   
1              0.04                      -429.0                       -97.0   
2              0.08                       -78.0                        31.0   
3              0.12                        14.0                       -81.0   
4              0.16                       377.0                      -104.0   

   Accelerometer - Spindle -X  Accelerometer - X Driving axle +Z  \
0                      -190.0                              -12.0   
1                         3.0                              -12.0   
2                      -285.0                              -10.0   
3                      -240.0                              -12.0   
4                       -84.0                              -12.0   

   Accelerometer - X Driving axle -X  Accelerometer - Y Driving 

In [4]:
#instalar tsfrech
!pip install tsfresh

In [5]:
#Librerias
import zipfile
import pandas as pd
import numpy as np
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import EfficientFCParameters
import os
import time

print("Librerías cargadas")

Librerías cargadas


In [6]:
#Downsampling para archivos de prueba
import zipfile
import pandas as pd
import numpy as np
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import EfficientFCParameters
import time

ruta_zip = '/content/drive/MyDrive/raw_data.zip'

with zipfile.ZipFile(ruta_zip, 'r') as z:
    todos = [a for a in z.namelist() if a.endswith('.csv')]
    archivos_prueba = todos[:5]

print(f"Procesando {len(archivos_prueba)} archivos de prueba...\n")

# 🔧 Número de puntos a los que reducimos cada señal
N_PUNTOS = 5000  # 5000 puntos por sensor por ciclo

resultados = []
inicio = time.time()

for i, nombre_archivo in enumerate(archivos_prueba):
    print(f"[{i+1}/{len(archivos_prueba)}] Procesando: {nombre_archivo}")

    try:
        with zipfile.ZipFile(ruta_zip, 'r') as z:
            with z.open(nombre_archivo) as f:
                df = pd.read_csv(f)

        # Eliminar timestamps
        df_senales = df.drop(columns=['Timestamps - Acc', 'Timestamps - Current'], errors='ignore')

        # 🔧 DOWNSAMPLING: reducir cada señal a N_PUNTOS
        n_original = len(df_senales)
        if n_original > N_PUNTOS:
            # Tomar índices espaciados uniformemente
            indices = np.linspace(0, n_original - 1, N_PUNTOS, dtype=int)
            df_senales = df_senales.iloc[indices].reset_index(drop=True)

        print(f"  Filas originales: {n_original}, después de downsampling: {len(df_senales)}")

        # Convertir a formato largo
        df_largo = df_senales.reset_index().melt(
            id_vars=['index'],
            var_name='sensor',
            value_name='value'
        )
        df_largo.rename(columns={'index': 'time'}, inplace=True)
        df_largo['id'] = nombre_archivo

        # Eliminar NaN y valores no finitos
        df_largo = df_largo.dropna(subset=['value'])
        df_largo = df_largo[np.isfinite(df_largo['value'])]

        print(f"  Filas válidas: {len(df_largo)}, sensores: {df_largo['sensor'].nunique()}")

        # Extraer features con tsfresh
        features = extract_features(
            df_largo,
            column_id='id',
            column_sort='time',
            column_kind='sensor',
            column_value='value',
            default_fc_parameters=EfficientFCParameters(),
            disable_progressbar=True,
            n_jobs=1
        )

        features = impute(features)
        resultados.append(features)
        print(f"  ✅ Features extraídas: {features.shape[1]}\n")

    except Exception as e:
        print(f"  ❌ Error: {e}\n")

if resultados:
    df_features = pd.concat(resultados)
    print(f"\n✅ Total: {df_features.shape[0]} ciclos, {df_features.shape[1]} features")
    print(f"⏱️ Tiempo total: {(time.time() - inicio)/60:.1f} minutos")
else:
    print("No se generaron features")

Procesando 5 archivos de prueba...

[1/5] Procesando: raw_data/P002_F01_C1.csv


  Filas originales: 2698201, después de downsampling: 5000
  Filas válidas: 41200, sensores: 20


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(


  ✅ Features extraídas: 15540

[2/5] Procesando: raw_data/P002_F01_C2.csv


  Filas originales: 1590701, después de downsampling: 5000
  Filas válidas: 41200, sensores: 20


...

/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(


  ✅ Features extraídas: 15540

[5/5] Procesando: raw_data/P002_F02_C1.csv


  Filas originales: 2804801, después de downsampling: 5000
  Filas válidas: 41200, sensores: 20


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(


  ✅ Features extraídas: 15540


✅ Total: 5 ciclos, 15540 features
⏱️ Tiempo total: 1.3 minutos


In [9]:
#Procesar 30 archivos de muestras
import zipfile
import pandas as pd
import numpy as np
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import EfficientFCParameters
import time

ruta_zip = '/content/drive/MyDrive/raw_data.zip'

with zipfile.ZipFile(ruta_zip, 'r') as z:
    todos = [a for a in z.namelist() if a.endswith('.csv')]
    archivos_muestra = todos[:30]  # primeros 30

print(f"Procesando {len(archivos_muestra)} archivos...\n")

N_PUNTOS = 5000
resultados = []
inicio = time.time()

for i, nombre_archivo in enumerate(archivos_muestra):
    print(f"[{i+1}/{len(archivos_muestra)}] {nombre_archivo}")
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as z:
            with z.open(nombre_archivo) as f:
                df = pd.read_csv(f)

        df_senales = df.drop(columns=['Timestamps - Acc', 'Timestamps - Current'], errors='ignore')

        if len(df_senales) > N_PUNTOS:
            indices = np.linspace(0, len(df_senales) - 1, N_PUNTOS, dtype=int)
            df_senales = df_senales.iloc[indices].reset_index(drop=True)

        df_largo = df_senales.reset_index().melt(
            id_vars=['index'], var_name='sensor', value_name='value'
        )
        df_largo.rename(columns={'index': 'time'}, inplace=True)
        df_largo['id'] = nombre_archivo

        df_largo = df_largo.dropna(subset=['value'])
        df_largo = df_largo[np.isfinite(df_largo['value'])]

        features = extract_features(
            df_largo, column_id='id', column_sort='time',
            column_kind='sensor', column_value='value',
            default_fc_parameters=EfficientFCParameters(),
            disable_progressbar=True, n_jobs=1
        )
        features = impute(features)
        resultados.append(features)
    except Exception as e:
        print(f"  ❌ Error: {e}")

# Concatenar
df_muestra = pd.concat(resultados)
print(f"\n✅ Muestra: {df_muestra.shape}")
print(f"⏱️ Tiempo: {(time.time() - inicio)/60:.1f} minutos")

# Guardar la muestra
df_muestra.to_csv('/content/drive/MyDrive/muestra_30_ciclos.csv')
print("💾 Guardada en Drive: muestra_30_ciclos.csv")

Procesando 30 archivos...

[1/30] raw_data/P002_F01_C1.csv


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(


[2/30] raw_data/P002_F01_C2.csv


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(


[3/30] raw_data/P002_F01_C3.csv


...

[29/30] raw_data/P002_F08_C2.csv


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(


[30/30] raw_data/P002_F08_C3.csv


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(



✅ Muestra: (30, 15540)
⏱️ Tiempo: 7.4 minutos
💾 Guardada en Drive: muestra_30_ciclos.csv


In [10]:
#Limpiar y seleccionar
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

# Cargar la muestra
df_muestra = pd.read_csv('/content/drive/MyDrive/muestra_30_ciclos.csv', index_col=0)

print(f"Features originales: {df_muestra.shape[1]}")

# 1. Eliminar features constantes (todos los valores iguales)
df_muestra = df_muestra.loc[:, df_muestra.nunique() > 1]
print(f"Después de eliminar constantes: {df_muestra.shape[1]}")

# 2. Eliminar features con todos los valores cero
df_muestra = df_muestra.loc[:, (df_muestra != 0).any(axis=0)]
print(f"Después de eliminar todo-cero: {df_muestra.shape[1]}")

# 3. Eliminar features con más del 90% de valores faltantes
df_muestra = df_muestra.loc[:, df_muestra.notna().mean() > 0.9]
print(f"Después de eliminar muchas-NaN: {df_muestra.shape[1]}")

# 4. Eliminar features con muy baja varianza
df_muestra_limpio = df_muestra.fillna(0)
selector = VarianceThreshold(threshold=0.01)
selector.fit(df_muestra_limpio)
columnas_utiles = df_muestra.columns[selector.get_support()].tolist()

print(f"\n✅ Features útiles finales: {len(columnas_utiles)}")

# Guardar la lista de columnas útiles
pd.Series(columnas_utiles).to_csv('/content/drive/MyDrive/columnas_utiles.csv', index=False, header=['feature'])
print("💾 Lista guardada en Drive: columnas_utiles.csv")

Features originales: 15540
Después de eliminar constantes: 12030
Después de eliminar todo-cero: 12030
Después de eliminar muchas-NaN: 12030

✅ Features útiles finales: 10727
💾 Lista guardada en Drive: columnas_utiles.csv


Los archivos crudos se llaman:

P002_F01_C1.csv

P002_F01_C2.csv

etc.

Pero en el CSV procesado (el que tiene la etiqueta CycleToFailureNormalized), las filas tienen columnas como NumberOfCycle, SampleIndex, TollIndex.

Necesitamos saber qué archivo crudo corresponde a qué fila del CSV procesado. Sin eso, no podemos asignar la etiqueta CycleToFailureNormalized a las features extraídas.

In [12]:
#Resolucion del mapeo
import zipfile
import pandas as pd
import re

ruta_zip = '/content/drive/MyDrive/raw_data.zip'
url = 'https://raw.githubusercontent.com/Coobaltoo13/milling-tool-life-fusion/main/data/raw/FeatureAndMetadata_Milling.csv'

# 1. Parsear los nombres de archivo
with zipfile.ZipFile(ruta_zip, 'r') as z:
    archivos = [a for a in z.namelist() if a.endswith('.csv')]

data = []
for a in archivos:
    match = re.search(r'P(\d+)_F(\d+)_C(\d+)', a)
    if match:
        data.append({'file': a, 'P': int(match.group(1)),
                     'F': int(match.group(2)), 'C': int(match.group(3))})

df_files = pd.DataFrame(data)
print("=== Archivos crudos ===")
print(f"Total: {len(df_files)}")
print(f"P (piezas): {sorted(df_files['P'].unique())}")
print(f"F (herramientas): {sorted(df_files['F'].unique())}")
print(f"C (ciclos): {sorted(df_files['C'].unique())}")
print(df_files.head(10))

# 2. Cargar el CSV procesado con los nombres correctos
df_proc = pd.read_csv(url, sep=';', header=0, decimal=',', skiprows=[1])
print(f"\n=== CSV procesado ===")
print(f"Total filas: {len(df_proc)}")

# Renombrar columnas clave para claridad
df_proc = df_proc.rename(columns={
    'Column122': 'NumberOfCycle',
    'Column123': 'SampleIndex',
    'Column124': 'TollIndex',
    'Column131': 'CycleToFailureNormalized'
})

print("\nPrimeras 10 filas (metadatos y etiqueta):")
print(df_proc[['NumberOfCycle', 'SampleIndex', 'TollIndex', 'CycleToFailureNormalized']].head(10))

print("\nÚltimas 10 filas:")
print(df_proc[['NumberOfCycle', 'SampleIndex', 'TollIndex', 'CycleToFailureNormalized']].tail(10))

# 3. Ver si el orden coincide
print("\n=== Comparación ===")
print("Primeros archivos crudos:")
print(df_files[['P', 'F', 'C']].head(10).to_string())
print("\nPrimeras filas del CSV procesado:")
print(df_proc[['SampleIndex', 'TollIndex', 'NumberOfCycle']].head(10).to_string())

=== Archivos crudos ===
Total: 968
P (piezas): [np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(23), np.int64(26), np.int64(33), np.int64(36), np.int64(38), np.int64(41), np.int64(108), np.int64(110), np.int64(112), np.int64(114), np.int64(119)]
F (herramientas): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
C (ciclos): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
                       file  P  F  C
0  raw_data/P002_F01_C1.csv  2  1  1
1  raw_data/P002_F01_C2.csv  2  1  2
2  raw_data/P002_F01_C3.csv  2  1  3
3  raw_data/P002_F01_C4.csv  2  1  4
4  raw_data/P002_F02_C1.csv  2  2  1
5  raw_data/P002_F02_C2.csv  2  2  2
6  raw_data/P002_F02_C3.csv  2  2  3
7  raw_data/P002_F02_C4.csv  2  2  4
8 

In [13]:
#Entender la relacion
import zipfile
import pandas as pd
import re

ruta_zip = '/content/drive/MyDrive/raw_data.zip'
url = 'https://raw.githubusercontent.com/Coobaltoo13/milling-tool-life-fusion/main/data/raw/FeatureAndMetadata_Milling.csv'

# 1. Cargar archivos crudos y ordenarlos
with zipfile.ZipFile(ruta_zip, 'r') as z:
    archivos = [a for a in z.namelist() if a.endswith('.csv')]

data = []
for a in archivos:
    match = re.search(r'P(\d+)_F(\d+)_C(\d+)', a)
    if match:
        data.append({'file': a, 'P': int(match.group(1)),
                     'F': int(match.group(2)), 'C': int(match.group(3))})

df_files = pd.DataFrame(data).sort_values(['P', 'F', 'C']).reset_index(drop=True)

# 2. Cargar CSV procesado
df_proc = pd.read_csv(url, sep=';', header=0, decimal=',', skiprows=[1])
df_proc = df_proc.rename(columns={
    'Column122': 'NumberOfCycle',
    'Column123': 'SampleIndex',
    'Column124': 'TollIndex',
    'Column131': 'CycleToFailureNormalized'
})

# 3. Ver la distribución de SampleIndex y TollIndex
print("=== Distribución de SampleIndex ===")
print(df_proc['SampleIndex'].value_counts().sort_index().head(20))

print("\n=== Distribución de TollIndex ===")
print(df_proc['TollIndex'].value_counts().sort_index().head(20))

print("\n=== Primeras 20 filas del CSV procesado ===")
print(df_proc[['SampleIndex', 'TollIndex', 'NumberOfCycle', 'CycleToFailureNormalized']].head(20).to_string())

print("\n=== Primeros 20 archivos crudos ordenados ===")
print(df_files.head(20).to_string())

# 4. Ver si hay algún patrón
print("\n=== ¿Cuántas filas por SampleIndex? ===")
print(df_proc.groupby('SampleIndex').size().head(20))

=== Distribución de SampleIndex ===
SampleIndex
2      46
3      42
6      40
7      43
8      10
10     23
11     28
12     84
14     84
15     42
16     20
17     41
23     32
26     42
33     65
36     19
38     42
41     42
108    81
110    83
Name: count, dtype: int64

=== Distribución de TollIndex ===
TollIndex
2       48
3       44
4       41
5       51
6       28
7      124
8      116
9      149
10      15
11       1
101     84
102     82
103     60
105    125
Name: count, dtype: int64

=== Primeras 20 filas del CSV procesado ===
    SampleIndex  TollIndex  NumberOfCycle  CycleToFailureNormalized
0             2          2              1                  1.000000
1             2          2              2                  0.979592
2             2          2              3                  0.959184
3             2          2              4                  0.938776
4             2          2              5                  0.918367
5             2          2              6       

Lo que descubrimos

Archivos crudos	CSV procesado

P (piezas): 24 valores únicos	SampleIndex: 24 valores únicos (coinciden)

F (herramientas): 1-12	TollIndex: valores como 2, 105... (no coincide
directamente)

C (ciclos): 1-7	NumberOfCycle: 1-125 (contador global)

La teoría más probable: El CSV se generó procesando los archivos crudos en el mismo orden. Es decir, la fila 0 del CSV corresponde al archivo 0 de la lista ordenada, la fila 1 al archivo 1, etc.




In [14]:
#Verificacion rapida
import zipfile
import pandas as pd
import re

ruta_zip = '/content/drive/MyDrive/raw_data.zip'
url = 'https://raw.githubusercontent.com/Coobaltoo13/milling-tool-life-fusion/main/data/raw/FeatureAndMetadata_Milling.csv'

# Archivos crudos ordenados
with zipfile.ZipFile(ruta_zip, 'r') as z:
    archivos = sorted([a for a in z.namelist() if a.endswith('.csv')])

# CSV procesado
df_proc = pd.read_csv(url, sep=';', header=0, decimal=',', skiprows=[1])
df_proc = df_proc.rename(columns={'Column123': 'SampleIndex', 'Column131': 'CycleToFailureNormalized'})

# Comparar
print("=== Verificación de orden ===\n")
for i in [0, 1, 2, 10, 100, 500, 967]:
    match = re.search(r'P(\d+)_F(\d+)_C(\d+)', archivos[i])
    p, f, c = int(match.group(1)), int(match.group(2)), int(match.group(3))
    sample = int(df_proc.iloc[i]['SampleIndex'])
    label = df_proc.iloc[i]['CycleToFailureNormalized']
    match_str = "✅" if p == sample else "❌"
    print(f"Fila {i}: Archivo={archivos[i]} → P={p}, F={f}, C={c}")
    print(f"         CSV → SampleIndex={sample}, Etiqueta={label:.4f}  {match_str}\n")

=== Verificación de orden ===

Fila 0: Archivo=raw_data/P002_F01_C1.csv → P=2, F=1, C=1
         CSV → SampleIndex=2, Etiqueta=1.0000  ✅

Fila 1: Archivo=raw_data/P002_F01_C2.csv → P=2, F=1, C=2
         CSV → SampleIndex=2, Etiqueta=0.9796  ✅

Fila 2: Archivo=raw_data/P002_F01_C3.csv → P=2, F=1, C=3
         CSV → SampleIndex=2, Etiqueta=0.9592  ✅

Fila 10: Archivo=raw_data/P002_F03_C3.csv → P=2, F=3, C=3
         CSV → SampleIndex=2, Etiqueta=0.7959  ✅

Fila 100: Archivo=raw_data/P006_F02_C6.csv → P=6, F=2, C=6
         CSV → SampleIndex=6, Etiqueta=0.9048  ✅

Fila 500: Archivo=raw_data/P017_F06_C4.csv → P=17, F=6, C=4
         CSV → SampleIndex=17, Etiqueta=0.0317  ✅

Fila 967: Archivo=raw_data/P119_F07_C1.csv → P=119, F=7, C=1
         CSV → SampleIndex=119, Etiqueta=0.0000  ✅



In [16]:
#Prueba con 50 archivos
import zipfile
import pandas as pd
import numpy as np
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import EfficientFCParameters
import time
import os
import re

# ===== CONFIGURACIÓN =====
ruta_zip = '/content/drive/MyDrive/raw_data.zip'
carpeta_salida = '/content/drive/MyDrive/features_tsfresh_lotes'
os.makedirs(carpeta_salida, exist_ok=True)

N_PUNTOS = 5000
GUARDAR_CADA = 50

# ===== CARGAR LISTA DE FEATURES ÚTILES =====
columnas_utiles = pd.read_csv('/content/drive/MyDrive/columnas_utiles.csv')['feature'].tolist()
print(f"Features útiles a conservar: {len(columnas_utiles)}")

# ===== CARGAR ETIQUETAS DEL CSV PROCESADO =====
url = 'https://raw.githubusercontent.com/Coobaltoo13/milling-tool-life-fusion/main/data/raw/FeatureAndMetadata_Milling.csv'
df_proc = pd.read_csv(url, sep=';', header=0, decimal=',', skiprows=[1])
etiquetas = df_proc['Column131'].values  # CycleToFailureNormalized
print(f"Etiquetas cargadas: {len(etiquetas)}")

# ===== LISTA DE ARCHIVOS ORDENADOS =====
with zipfile.ZipFile(ruta_zip, 'r') as z:
    archivos = sorted([a for a in z.namelist() if a.endswith('.csv')])
print(f"Total archivos: {len(archivos)}")

# ===== PRUEBA: SOLO PRIMEROS 50 ARCHIVOS =====
inicio_idx = 0
fin_idx = 50  # Solo los primeros 50
print(f"\n🔬 MODO PRUEBA: procesando archivos 0 a {fin_idx}\n")

# ===== BUCLE PRINCIPAL (SOLO UN LOTE) =====
for lote_inicio in range(inicio_idx, fin_idx, GUARDAR_CADA):
    lote_fin = min(lote_inicio + GUARDAR_CADA, fin_idx, len(archivos))
    archivos_lote = archivos[lote_inicio:lote_fin]
    etiquetas_lote = etiquetas[lote_inicio:lote_fin]

    print(f"\n===== Lote {lote_inicio // GUARDAR_CADA + 1}: archivos {lote_inicio+1} a {lote_fin} =====")

    resultados = []
    inicio = time.time()

    for i, (nombre_archivo, etiqueta) in enumerate(zip(archivos_lote, etiquetas_lote)):
        print(f"  [{i+1}/{len(archivos_lote)}] {nombre_archivo}")
        try:
            with zipfile.ZipFile(ruta_zip, 'r') as z:
                with z.open(nombre_archivo) as f:
                    df = pd.read_csv(f)

            df_senales = df.drop(columns=['Timestamps - Acc', 'Timestamps - Current'], errors='ignore')

            if len(df_senales) > N_PUNTOS:
                indices = np.linspace(0, len(df_senales) - 1, N_PUNTOS, dtype=int)
                df_senales = df_senales.iloc[indices].reset_index(drop=True)

            df_largo = df_senales.reset_index().melt(
                id_vars=['index'], var_name='sensor', value_name='value'
            )
            df_largo.rename(columns={'index': 'time'}, inplace=True)
            df_largo['id'] = nombre_archivo

            df_largo = df_largo.dropna(subset=['value'])
            df_largo = df_largo[np.isfinite(df_largo['value'])]

            features = extract_features(
                df_largo, column_id='id', column_sort='time',
                column_kind='sensor', column_value='value',
                default_fc_parameters=EfficientFCParameters(),
                disable_progressbar=True, n_jobs=1
            )
            features = impute(features)

            # Filtrar solo las features útiles
            features_filtradas = features[[c for c in columnas_utiles if c in features.columns]]

            # Agregar etiqueta
            features_filtradas['CycleToFailureNormalized'] = etiqueta
            features_filtradas['archivo'] = nombre_archivo

            resultados.append(features_filtradas)

        except Exception as e:
            print(f"  ❌ Error en {nombre_archivo}: {e}")

    if resultados:
        df_lote = pd.concat(resultados)
        ruta_lote = os.path.join(carpeta_salida, f'batch_prueba.csv')
        df_lote.to_csv(ruta_lote)
        print(f"\n✅ Lote de prueba guardado: {ruta_lote}")
        print(f"   Forma: {df_lote.shape}")
        print(f"⏱️ Tiempo: {(time.time() - inicio)/60:.1f} min")

print("\n🎉 Prueba completada. Revisa el archivo 'batch_prueba.csv' en Drive.")

Features útiles a conservar: 10727
Etiquetas cargadas: 968
Total archivos: 968

🔬 MODO PRUEBA: procesando archivos 0 a 50


===== Lote 1: archivos 1 a 50 =====
  [1/50] raw_data/P002_F01_C1.csv


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(
/tmp/ipykernel_2903/1797672885.py:85: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features_filtradas['CycleToFailureNo

  [2/50] raw_data/P002_F01_C2.csv


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(
/tmp/ipykernel_2903/1797672885.py:85: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features_filtradas['CycleToFailureNo

  [3/50] raw_data/P002_F01_C3.csv


...

  [49/50] raw_data/P003_F01_C3.csv


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(
/tmp/ipykernel_2903/1797672885.py:85: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features_filtradas['CycleToFailureNo

  [50/50] raw_data/P003_F01_C4.csv


/usr/local/lib/python3.13/dist-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns ['Accelerometer - Spindle +Y__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -X__query_similarity_count__query_None__threshold_0.0'
 'Accelerometer - Spindle -Z__query_similarity_count__query_None__threshold_0.0'
 ... 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_98'
 'Current - Spindle L3__fft_coefficient__attr_"angle"__coeff_99'
 'Current - Spindle L3__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(
/tmp/ipykernel_2903/1797672885.py:85: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features_filtradas['CycleToFailureNo


✅ Lote de prueba guardado: /content/drive/MyDrive/features_tsfresh_lotes/batch_prueba.csv
   Forma: (50, 10729)
⏱️ Tiempo: 12.4 min

🎉 Prueba completada. Revisa el archivo 'batch_prueba.csv' en Drive.
